# Random Forest - Model Training

Entrenamiento de modelos Random Forest (BALANCED vs RAW) con GridSearchCV y métricas en training set.

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported")

 Libraries imported


## 1. Configure Paths

In [2]:
# Define paths
BASE_PATH = Path('../../..').resolve()
DATA_PATH = BASE_PATH / 'data' / 'processed' / 'pickle'
MODELS_PATH = BASE_PATH / 'models' / 'random_forest'
RESULTS_PATH = BASE_PATH / 'data' / 'results' / 'random_forest'

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Paths configured")
print(f"  Models  → {MODELS_PATH}")
print(f"  Results → {RESULTS_PATH}")

✓ Paths configured
  Models  → /home/pablo/Desktop/Estressss/pdg/models/random_forest
  Results → /home/pablo/Desktop/Estressss/pdg/data/results/random_forest


## 2. Load Data

In [3]:
# Load training data (BALANCED and RAW)
with open(DATA_PATH / 'X_train_balanced.pkl', 'rb') as f:
    X_train_balanced = pickle.load(f)
with open(DATA_PATH / 'y_train_balanced.pkl', 'rb') as f:
    y_train_balanced = pickle.load(f)

with open(DATA_PATH / 'X_train_raw.pkl', 'rb') as f:
    X_train_raw = pickle.load(f)
with open(DATA_PATH / 'y_train_raw.pkl', 'rb') as f:
    y_train_raw = pickle.load(f)

# Convert to numpy arrays if needed
if isinstance(X_train_balanced, pd.DataFrame):
    X_train_balanced = X_train_balanced.values
if isinstance(X_train_raw, pd.DataFrame):
    X_train_raw = X_train_raw.values

# Ensure targets are Series
if isinstance(y_train_balanced, pd.DataFrame):
    y_train_balanced= y_train_balanced.iloc[:, 0]
if isinstance(y_train_raw, pd.DataFrame):
    y_train_raw = y_train_raw.iloc[:, 0]

print(f"✓ BALANCED Training set : {X_train_balanced.shape}")
print(f"✓ RAW   Training set : {X_train_raw.shape}")
print(f"Class distribution:")
print(f"  BALANCED Train : {dict(pd.Series(y_train_balanced).value_counts().sort_index())}")
print(f"  RAW   Train : {dict(pd.Series(y_train_raw).value_counts().sort_index())}")

✓ BALANCED Training set : (898, 651)
✓ RAW   Training set : (699, 651)
Class distribution:
  BALANCED Train : {0: np.int64(633), 1: np.int64(265)}
  RAW   Train : {0: np.int64(633), 1: np.int64(66)}


## 3. Hyperparameter Tuning - Grid Search

> **Métrica objetivo:**  — priorizamos detectar muertes.
> 
> **Nota:** Random Forest no tiene  ni  como XGBoost. Sus hiperparámetros clave son , , ,  y .

In [4]:
print("=" * 70)
print("HYPERPARAMETER TUNING - GRID SEARCH")
print("=" * 70)
print("Configuration:")
print("  • Metric: Recall Macro (10-fold Stratified CV)")
print("  • Cross-validation: StratifiedKFold (n_splits=10)")
print("  • Strategy: Tune n_estimators, max_depth, min_samples_split,")
print("              min_samples_leaf, max_features")

# CV strategy — same as XGBoost for fair comparison
cv_stratified = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Parameter grid — specific to Random Forest
# (RF no tiene learning_rate, gamma ni colsample_bytree)
param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2'],
    'criterion':         ['gini', 'entropy'], # Añadido
}

print(f"Parameter grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

HYPERPARAMETER TUNING - GRID SEARCH
Configuration:
  • Metric: Recall Macro (10-fold Stratified CV)
  • Cross-validation: StratifiedKFold (n_splits=10)
  • Strategy: Tune n_estimators, max_depth, min_samples_split,
              min_samples_leaf, max_features
Parameter grid:
  n_estimators: [100, 200, 300]
  max_depth: [None, 5, 10, 20]
  min_samples_split: [2, 5, 10]
  min_samples_leaf: [1, 2, 4]
  max_features: ['sqrt', 'log2']
  criterion: ['gini', 'entropy']


In [ ]:
'''
# ======================================================================
# ENTRENAMIENTO MANUAL DIRECTO (Sin GridSearchCV)
# ======================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

print("Entrenando modelo BALANCED con los mejores hiperparámetros...")
# 1. Creamos y entrenamos el modelo para BALANCED directamente
model_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=4,
    min_samples_split=10,
    max_features='log2',
    criterion='gini',
    max_samples=0.7,
    class_weight='balanced',
    bootstrap=True,
    oob_score=True,  # Activamos OOB
    random_state=42,
    n_jobs=-1
)
sample_weights_balanced = compute_sample_weight('balanced', y_train_balanced)
model_balanced.fit(X_train_balanced, y_train_balanced, sample_weight=sample_weights_balanced)
print(f"✓ Modelo BALANCED entrenado (OOB Score: {model_balanced.oob_score_:.4f})")

print("-" * 50)

print("Entrenando modelo RAW con los mejores hiperparámetros...")
# 2. Creamos y entrenamos el modelo para RAW directamente
model_raw = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=4,
    min_samples_split=10,
    max_features='log2',
    max_samples=0.7,
    criterion='entropy',
    class_weight='balanced',
    bootstrap=True,
    oob_score=True,  # Activamos OOB
    random_state=42,
    n_jobs=-1
)
sample_weights_raw = compute_sample_weight('balanced', y_train_raw)
model_raw.fit(X_train_raw, y_train_raw, sample_weight=sample_weights_raw)
print(f"✓ Modelo RAW entrenado (OOB Score: {model_raw.oob_score_:.4f})")

print("=" * 50)
print("¡Ambos modelos entrenados exitosamente en segundos!")
'''

Entrenando modelo BALANCED con los mejores hiperparámetros...
✓ Modelo BALANCED entrenado (OOB Score: 0.4376)
--------------------------------------------------
Entrenando modelo RAW con los mejores hiperparámetros...
✓ Modelo RAW entrenado (OOB Score: 0.0944)
¡Ambos modelos entrenados exitosamente en segundos!


## 4. Base Model and Grid Search Function

In [ ]:
def create_base_model():
    """Base Random Forest model with fixed parameters.
    
    class_weight='balanced' le indica al modelo que compense
    automáticamente el desbalance de clases durante el entrenamiento.
    Esto es importante porque nuestro dataset tiene muchas más muestras
    de clase 0 que de clase 1 o 2.
    """
    return RandomForestClassifier(
        class_weight='balanced',   # compensa desbalance internamente
        max_samples=0.7,
        bootstrap=True,
        random_state=42,
        oob_score=True,
        n_jobs=-1,                  # usa todos los núcleos disponibles
    )


def run_grid_search(X_train, y_train, model_name):
    """Execute GridSearchCV for BALANCED or RAW model.
    
    Usamos sample_weight='balanced' además de class_weight en el modelo
    para reforzar el peso de las clases minoritarias durante la búsqueda.
    """
    print(f"{'-'*70}")
    print(f"Grid Search: {model_name} Model")
    print(f"{'-'*70}")

    # compute_sample_weight asigna más peso a las clases con menos muestras
    sample_weights = compute_sample_weight('balanced', y_train)

    grid_search = GridSearchCV(
        estimator=create_base_model(),
        param_grid=param_grid,
        cv=cv_stratified,
        scoring='recall_macro',    # priorizar detección de mortalidad
        n_jobs=-1,
        verbose=1,
    )

    print(f"Searching optimal hyperparameters (this may take a few minutes)...")
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)

    print(f"✓ Best hyperparameters ({model_name}):")
    for param, value in grid_search.best_params_.items():
        print(f"    {param}: {value}")

    print(f"✓ Best Recall Macro (CV): {grid_search.best_score_:.4f}")

    # Save CV results to CSV for later analysis
    cv_results = pd.DataFrame(grid_search.cv_results_)
    suffix = 'balanced' if 'BALANCED' in model_name else 'raw'
    results_path = RESULTS_PATH / f'gridsearch_results_{suffix}.csv'
    cv_results.to_csv(results_path, index=False)
    print(f"✓ Grid Search results saved → {results_path.name}")

    return grid_search.best_estimator_, grid_search

## 5. Run Grid Search for Both Models

In [ ]:
# Run grid search — BALANCED model
model_balanced, gs_balanced = run_grid_search(X_train_balanced, y_train_balanced, 'BALANCED')

# Run grid search — RAW model
model_raw, gs_raw = run_grid_search(X_train_raw, y_train_raw, 'RAW')

print(f"" + "="*70)
print(f"✓ Both models trained successfully")
print(f"="*70)

## 6. Training Set Performance

> ⚠️ Estas métricas son sobre el **training set**. Si son perfectas (1.0000) es normal en RF — el modelo memoriza los datos de entrenamiento. Lo que importa es el **test set**, que se evalúa en el notebook de evaluación.

In [7]:
print("" + "="*70)
print("TRAINING SET PERFORMANCE")
print("="*70)

#BALANCED Model on training data
y_train_pred_balanced = model_balanced.predict(X_train_balanced)
print(f"BALANCED Model:")
print(f"  Accuracy : {accuracy_score(y_train_balanced, y_train_pred_balanced):.4f}")
print(f"  Precision: {precision_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")

# RAW Model on training data
y_train_pred_raw = model_raw.predict(X_train_raw)
print(f"RAW Model:")
print(f"  Accuracy : {accuracy_score(y_train_raw, y_train_pred_raw):.4f}")
print(f"  Precision: {precision_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")

TRAINING SET PERFORMANCE
BALANCED Model:
  Accuracy : 0.4510
  Precision: 0.6748
  Recall   : 0.6106
  F1-Score : 0.4402
RAW Model:
  Accuracy : 0.0944
  Precision: 0.0472
  Recall   : 0.5000
  F1-Score : 0.0863


## 7. Per-Class Metrics (Training Set)

In [8]:
print("" + "="*70)
print("PER-CLASS METRICS (TRAINING SET)")
print("="*70)

# BALANCED per-class
prec_s, rec_s, f1_s, sup_s = precision_recall_fscore_support(
    y_train_balanced, y_train_pred_balanced, average=None
)
print(f"BALANCED Model:")
df_train_balanced = pd.DataFrame({
    'Class'    : [0, 1],
    'Precision': prec_s,
    'Recall'   : rec_s,
    'F1-Score' : f1_s,
    'Support'  : sup_s
})
print(df_train_balanced.to_string(index=False))

# RAW per-class
prec_r, rec_r, f1_r, sup_r = precision_recall_fscore_support(
    y_train_raw, y_train_pred_raw, average=None
)
print(f"RAW Model:")
df_train_raw = pd.DataFrame({
    'Class'    : [0, 1],
    'Precision': prec_r,
    'Recall'   : rec_r,
    'F1-Score' : f1_r,
    'Support'  : sup_r
})
print(df_train_raw.to_string(index=False))

PER-CLASS METRICS (TRAINING SET)
BALANCED Model:
 Class  Precision   Recall  F1-Score  Support
     0   1.000000 0.221169  0.362225      633
     1   0.349604 1.000000  0.518084      265
RAW Model:
 Class  Precision  Recall  F1-Score  Support
     0   0.000000     0.0  0.000000      633
     1   0.094421     1.0  0.172549       66


## 8. Save Models

In [9]:
# Save BALANCED model
model_path_balanced = MODELS_PATH / 'modelo_random_forest_balanced.pkl'
with open(model_path_balanced, 'wb') as f:
    pickle.dump(model_balanced, f)
print(f"✓ BALANCED model saved → {model_path_balanced}")

# Save RAW model
model_path_raw = MODELS_PATH / 'modelo_random_forest_raw.pkl'
with open(model_path_raw, 'wb') as f:
    pickle.dump(model_raw, f)
print(f"✓ RAW model saved   → {model_path_raw}")

print(f"✅ Both models ready for evaluation")
print(f"   → Run random_forest_evaluation.ipynb next")

✓ BALANCED model saved → /home/pablo/Desktop/Estressss/pdg/models/random_forest/modelo_random_forest_balanced.pkl
✓ RAW model saved   → /home/pablo/Desktop/Estressss/pdg/models/random_forest/modelo_random_forest_raw.pkl
✅ Both models ready for evaluation
   → Run random_forest_evaluation.ipynb next
